# Single-Cell RNA-seq with TMAP

This notebook shows how to use TMAP for single-cell analysis using the PBMC 3k
dataset (the "hello world" of single-cell genomics). We demonstrate:

1. Fitting TMAP on PCA-reduced single-cell data
2. Coloring by cell type and gene expression
3. **Pseudotime**: computing tree distances as a proxy for biological progression
4. Side-by-side comparison with UMAP

**Requirements:** `pip install scanpy "tmap2[singlecell,notebook]"` (the `notebook` extra installs jupyter-scatter for the interactive views)


In [1]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc

from tmap import TMAP
from tmap.utils import from_anndata, marker_scores

warnings.filterwarnings("ignore", category=FutureWarning)


## Load the PBMC 3k dataset

Scanpy ships a pre-processed version with PCA, UMAP, and Louvain clusters
already computed. We use the PCA embedding (`adata.obsm['X_pca']`) as input
to TMAP. Scanpy's UMAP in this dataset was built from the same PCA representation.


In [2]:
adata = sc.datasets.pbmc3k_processed()
print(f"Cells: {adata.n_obs}, Genes: {adata.n_vars}")
print(f"PCA shape: {adata.obsm['X_pca'].shape}")
print(f"Cell types: {adata.obs['louvain'].unique().tolist()}")


  0%|          | 0.00/23.5M [00:00<?, ?B/s]

Cells: 2638, Genes: 1838
PCA shape: (2638, 50)
Cell types: ['CD4 T cells', 'B cells', 'CD14+ Monocytes', 'NK cells', 'CD8 T cells', 'FCGR3A+ Monocytes', 'Dendritic cells', 'Megakaryocytes']


## Fit TMAP

Three lines: create the estimator, fit on PCA coordinates, store the
embedding back into the AnnData object (the standard Scanpy convention).


In [3]:
model = TMAP(metric="euclidean", n_neighbors=15, seed=42)
model.fit(from_anndata(adata))  # adata.obsm["X_pca"] by default

# Store back into AnnData so Scanpy plotting functions can use it
adata.obsm["X_tmap"] = model.embedding_
print(f"Embedding shape: {model.embedding_.shape}")


Embedding shape: (2638, 2)


## Color by cell type

`model.to_tmapviz()` builds a viewer, and `add_color_layout` adds a color layer from
any metadata column. Here we color by the Louvain cluster labels (cell types).
The notebook widget draws points only and warns that the tree edges are ignored.
The HTML file written in the next cell includes them.


In [ ]:
viz = model.to_tmapviz()

viz.add_color_layout('Louvain', values=adata.obs["louvain"].to_list(), color="tab20", categorical=True)
viz.show()


In [ ]:
viz.write_html('single_cell_tmap.html')


## Color by gene expression

CST3 is a marker for monocytes and dendritic cells. We extract its expression
with `marker_scores` (the mean expression of a gene list per cell, here a
single gene) and pass it as a raw array to `color_by`.


In [5]:
cst3_expr = marker_scores(adata, ["CST3"])
model.plot(color_by=cst3_expr, color_map="viridis")


## Pseudotime via tree distances

**Pseudotime** is how biologists study differentiation trajectories.
Pick a "root" cell, compute tree distances to all other cells, and color by
this value. In a differentiation dataset the root is usually a stem or
progenitor cell. PBMC 3k has no annotated stem or progenitor cells, so here we
use a CD14+ monocyte as the root to show the mechanics. The colors then show
how far each cell sits from that monocyte along the tree.

`distances_from(root)` returns the tree distance from the root to every cell:
the sum of edge weights along the unique tree path. These distances are exact
for the tree. The tree itself is an approximation of the data: it keeps only
the minimum spanning tree edges of the kNN graph, so two cells can be farther
apart along the tree than in PCA space.


In [6]:
# Pick a CD14+ Monocyte as root (a differentiated myeloid cell)
mono_mask = adata.obs["louvain"] == "CD14+ Monocytes"
root_idx = int(np.where(mono_mask)[0][0])

pseudotime = model.distances_from(root_idx)
print(f"Root cell: {root_idx} ({adata.obs['louvain'].iloc[root_idx]})")
print(f"Pseudotime range: [{pseudotime.min():.2f}, {pseudotime.max():.2f}]")

model.plot(color_by=pseudotime, color_map="magma")


Root cell: 3 (CD14+ Monocytes)
Pseudotime range: [0.00, 349.77]


## UMAP vs TMAP comparison

Side-by-side matplotlib plots to compare the two embeddings.
Both use the same PCA input and the same cell type labels.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

cell_types = adata.obs["louvain"]
categories = cell_types.cat.categories
codes = cell_types.cat.codes.values
cmap = plt.cm.tab20

# UMAP (already in adata)
umap_coords = adata.obsm["X_umap"]
axes[0].scatter(umap_coords[:, 0], umap_coords[:, 1],
                c=codes, cmap=cmap, s=3, alpha=0.7)
axes[0].set_title("UMAP")
axes[0].set_xticks([])
axes[0].set_yticks([])

# TMAP
tmap_coords = model.embedding_
sc_tmap = axes[1].scatter(tmap_coords[:, 0], tmap_coords[:, 1],
                          c=codes, cmap=cmap, s=3, alpha=0.7)
axes[1].set_title("TMAP (tree layout)")
axes[1].set_xticks([])
axes[1].set_yticks([])

# Legend
handles = [plt.Line2D([0], [0], marker='o', color='w',
                       markerfacecolor=cmap(i / (len(categories) - 1)),
                       markersize=8, label=cat)
           for i, cat in enumerate(categories)]
fig.legend(handles=handles, loc="lower center", ncol=4,
           fontsize=9, frameon=False)

plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()
